In [ ]:
from notebook_utils import load_hydra_config, to_pil
from when2plan.envs.box_push.env import BoxPushEnv
from omegaconf import OmegaConf
import numpy as np
from IPython.display import display, clear_output
import numpy as np
import torch as th
import time
import copy
from when2plan.utils import collect_render_data_with_planner, get_model_free_metrics, plot_model_free_metrics
from when2plan.models.pretrain_model_free import pretrain_model_free_sub_env
from argparse import Namespace
from when2plan.envs.meta_env import MetaEnv

In [ ]:
in_distribution_cfg = load_hydra_config("../experiments/configs", "config", ["env=box_push", 'experiment=box_push_pretrain'])
out_of_distribution_cfg = load_hydra_config("../experiments/configs", "config", ["env=box_push", 'experiment=box_push_pretrain'])

num_tasks = 200
in_distribution_cfg.experiment.num_cached_tasks = num_tasks * 2
out_of_distribution_cfg.experiment.num_cached_tasks = num_tasks * 2

# Set in distribution vs out of distribution parameters
box_push_id = BoxPushEnv(in_distribution_cfg)
box_push_ood = BoxPushEnv(out_of_distribution_cfg)

box_push_id.set_cache_task_range(0, in_distribution_cfg.experiment.num_pretrained_tasks)
box_push_ood.set_cache_task_range(0, 20)  

_, _ = box_push_id.reset()
_, _ = box_push_ood.reset()

In [ ]:
# Pretrain model free agent on in-distribution environment
args = Namespace(
    sub_env="box_push",
    model_free_action_space="discrete",
    model_free_observation_space="tensor",
    num_actors=4,
    steps=10000,
    batch_size=64,
    epochs=70,
    base_seed=42,
    plot=True,
    save_actor=True,
)
pretrained_networks, env = pretrain_model_free_sub_env(args, box_push_id)

In [ ]:
# Create meta environments for in-distribution and out-of-distribution
id_meta_env = MetaEnv(in_distribution_cfg, box_push_id, pretrained_networks)
ood_meta_env = MetaEnv(out_of_distribution_cfg, box_push_ood, pretrained_networks)

# Get type of meta step from config
model_free_action = 0

metrics = get_model_free_metrics(id_meta_env, ood_meta_env, n_steps=5000, model_free_action=model_free_action)
plot_model_free_metrics(metrics, save_path='box_push_model_free_metrics.png')

In [ ]:
# sub_env render with planner

id_images = []
_, _ = box_push_id.reset(puzzle_idx=1)

# Get plan
for _ in range(100):
    id_images.append(box_push_id.render(scaling_factor=20))
    plan_result = box_push_id.get_plan(max_expansions=2000)
    if plan_result is None or len(plan_result[0]) == 0:
        break

    actions, _ = plan_result
    action = actions[0]
    _, _, term, trunc, _ = box_push_id.step(action)
    done = term or trunc
    if done:
        break

clear_output()
for img in id_images:
    display(img)
    time.sleep(0.1)
    clear_output()